In [1]:
import os
import sys
os.chdir("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import chi2, f_regression


df_listings = pd.read_csv('data/combined_csvs/listings_property_vals.csv')
df_future = pd.read_csv('data/combined_csvs/future_rates.csv')
df_past = pd.read_csv('data/combined_csvs/past_rates.csv')
df_reviews = pd.read_csv('data/combined_csvs/reviews.csv')

In [2]:
df_listings["amenities"]

0       Hair dryer,Body soap,Hot water,Essentials,Hang...
1       Hair dryer,Hot water,Essentials,Iron,TV,Air co...
2       Hair dryer,Shampoo,Hot water,Shower gel,Washer...
3       Hair dryer,Shampoo,Hot water,Washer,Dryer,Esse...
4       Hair dryer,Hot water,Washer,Dryer,Essentials,B...
                              ...                        
2877    TV,Cable TV,Blender,Wifi,Air conditioning,Kitc...
2878    Pets allowed,Washer,Dryer,Essentials,Iron,TV,A...
2879    Hair dryer,Shampoo,Hot water,Washer,Dryer,Esse...
2880    Pets allowed,Bathtub,Hair dryer,Shampoo,Condit...
2881    Hair dryer,Hot water,Washer,Dryer,Essentials,B...
Name: amenities, Length: 2882, dtype: str

In [3]:
amenities_encoded = df_listings['amenities'].str.get_dummies(sep=',')
amenities_encoded

,Air conditioning,Arcade games,BBQ grill,Baby bath,Baby monitor,Baby safety gates,Babysitter recommendations,Backyard,Baking sheet,Barbecue utensils,...,TV,Table corner guards,Theme room,Toaster,Trash compactor,Washer,Waterfront,Wifi,Window guards,Wine glasses
0,1,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,1,0,0
1,1,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,1,0,0
2,1,0,0,0,0,0,0,1,0,0,...,1,0,0,0,0,1,0,1,0,0
3,1,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,1,0,1,0,0
4,1,0,0,0,0,0,0,0,0,0,...,1,0,0,1,0,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2877,1,0,0,0,0,0,0,1,0,0,...,1,0,0,1,0,1,0,1,0,0
2878,1,0,0,0,0,0,0,1,0,1,...,1,0,0,1,0,1,0,1,0,0
2879,1,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,1,0,1,0,0
2880,1,0,0,0,0,0,0,0,1,0,...,1,0,0,0,0,1,0,1,0,0


In [4]:
len(amenities_encoded.columns)

135

In [5]:
from scipy.stats import mannwhitneyu

def should_keep_amenity(col, target, min_freq=0.05, max_freq=0.95, alpha=0.01):
    freq = col.mean()
    
    # 1. If it falls in the healthy variance zone, keep it automatically
    if min_freq <= freq <= max_freq:
        return True
    
    # 2. If it's too ubiquitous (e.g., > 98%), drop it. It offers no contrast.
    if freq > 0.98:
        return False
        
    # 3. If it's rare, check if the price distribution is significantly different
    # when the amenity is present vs. when it's absent
    price_with = target[col == 1]
    price_without = target[col == 0]
    
    # Ensure we have at least a few samples to test
    if len(price_with) < 5:
        return False
        
    # Non-parametric test to see if 'rare' actually moves the needle on price
    stat, p_val = mannwhitneyu(price_with, price_without, alternative='two-sided')
    
    # Keep if statistically significant
    return p_val < alpha

# Apply the conditional filter
keep_columns = [
    col for col in amenities_encoded.columns 
    if should_keep_amenity(amenities_encoded[col], df_listings['ttm_revenue']) # replacing target with your price column
]

amenities_filtered = amenities_encoded[keep_columns]

X = amenities_filtered
# y = df_listings['ttm_revenue']

# # f_regression scores the linear relationship of each binary feature vs continuous revenue
# f_stats, p_values = f_regression(X, y)

# # Pack into a clean DataFrame
# results = pd.DataFrame({
#     'amenity': X.columns,
#     'f_stat': f_stats,
#     'p_value': p_values
# })

# # Filter for statistically significant features (p < 0.05)
# significant_features = results[results['p_value'] < 0.05].sort_values(by='p_value')

# print(f"Found {len(significant_features)} significant amenities out of {len(X.columns)}.")
# print(significant_features.head(20))

# significant_features = list(significant_features["amenity"])
significant_features = amenities_filtered.columns

In [7]:
len(significant_features)

93